# F06 - Exploratory data analysis and visualization

**Hardware:** CPU-only · **Time:** 2-3 hours · **Prerequisite:** F05

Exploratory data analysis (EDA) is how you build a mental model of a dataset
before modeling. This notebook uses a synthetic house-price table and produces
three saved figures plus a short list of evidence-based observations.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
print("matplotlib", matplotlib.__version__)

matplotlib 3.11.2


In [2]:
def make_houses(n: int = 600, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    area = rng.normal(1800, 500, n).clip(600, 4000)
    bedrooms = rng.integers(1, 6, n)
    age = rng.integers(0, 60, n)
    distance = rng.gamma(2, 2, n).clip(0.5, 25)
    garage = rng.integers(0, 2, n)
    hood = rng.choice(["north", "south", "east", "west"], n, p=[0.3, 0.25, 0.25, 0.2])
    effect = pd.Series(hood).map({"north": 35000, "south": -10000, "east": 15000, "west": -5000}).to_numpy()
    price = (
        120000 + 95 * area + 18000 * bedrooms - 900 * age - 4500 * distance
        + 12000 * garage + 0.00004 * area**2 + effect + rng.normal(0, 20000, n)
    ).clip(50000, None)
    return pd.DataFrame(
        {
            "area_sqft": area.round(1),
            "bedrooms": bedrooms,
            "age_years": age,
            "distance_km": distance.round(2),
            "garage": garage,
            "neighbourhood": hood,
            "sale_price": price.round(0),
        }
    )


df = make_houses()
print("shape:", df.shape)
print(df.describe().round(2).to_string())
assert df.shape == (600, 7)

shape: (600, 7)
       area_sqft  bedrooms  age_years  distance_km  garage  sale_price
count     600.00    600.00     600.00       600.00  600.00      600.00
mean     1787.54      2.94      29.05         4.04    0.52   317660.92
std       485.24      1.41      17.12         2.96    0.50    62195.58
min       600.00      1.00       0.00         0.50    0.00   139901.00
25%      1451.10      2.00      15.00         1.92    0.00   275700.75
50%      1789.35      3.00      27.00         3.44    1.00   317513.50
75%      2093.70      4.00      44.00         5.32    1.00   356399.75
max      3256.90      5.00      59.00        17.49    1.00   507728.00


## 1. Univariate distributions

Check the target and the main numeric features before looking for
relationships. Ask: Is the distribution skewed? Are there impossible values?
Are there obvious clusters or gaps?

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].hist(df["sale_price"] / 1000, bins=25, color="#3b6ea5", edgecolor="white")
axes[0].set_title("Sale price (thousand USD)")
axes[1].hist(df["area_sqft"], bins=25, color="#4f8f6a", edgecolor="white")
axes[1].set_title("Area (sq ft)")
axes[2].hist(df["distance_km"], bins=25, color="#b06a3b", edgecolor="white")
axes[2].set_title("Distance to centre (km)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f06_distributions.png", bbox_inches="tight")
plt.close(fig)
print("saved", OUTPUT_DIR / "f06_distributions.png")
assert (OUTPUT_DIR / "f06_distributions.png").exists()

saved outputs\f06_distributions.png


## 2. Relationships and correlation

Correlation measures linear association, not causation. A strong correlation
between area and price is expected; a correlation between `age_years` and price
may be weaker or confounded by location. Inspect the scatter plot before trusting
a single number.

In [4]:
numeric = ["area_sqft", "bedrooms", "age_years", "distance_km", "garage", "sale_price"]
corr = df[numeric].corr()
print(corr["sale_price"].sort_values(ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(df["area_sqft"], df["sale_price"] / 1000, s=18, alpha=0.55, color="#3b6ea5")
ax.set_xlabel("Area (sq ft)")
ax.set_ylabel("Sale price (thousand USD)")
ax.set_title("Area and sale price")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f06_area_price.png", bbox_inches="tight")
plt.close(fig)
assert corr.loc["area_sqft", "sale_price"] > 0.5
print("area-price correlation:", round(corr.loc["area_sqft", "sale_price"], 3))

sale_price     1.000
area_sqft      0.749
bedrooms       0.384
garage         0.083
distance_km   -0.174
age_years     -0.226
area-price correlation: 0.749


## 3. Compare groups and check sample sizes

`neighbourhood` is categorical. A box plot shows the median and spread. The table
shows counts: a group with very few houses should not drive a conclusion.

In [5]:
summary = df.groupby("neighbourhood")["sale_price"].agg(["count", "median", "mean"]).sort_values("median")
print(summary.round(0).to_string())

fig, ax = plt.subplots(figsize=(6, 4))
order = summary.index.tolist()
data = [df.loc[df["neighbourhood"] == hood, "sale_price"] / 1000 for hood in order]
ax.boxplot(data, tick_labels=order)
ax.set_ylabel("Sale price (thousand USD)")
ax.set_title("Price by neighbourhood")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "f06_price_by_neighbourhood.png", bbox_inches="tight")
plt.close(fig)
assert summary["count"].min() >= 50
print("smallest group:", int(summary["count"].min()))

               count    median      mean
neighbourhood                           
south            135  297218.0  294867.0
west             103  301761.0  304318.0
east             162  323728.0  320141.0
north            200  337716.0  337910.0
smallest group: 103


## 4. Outliers and impossible values

The interquartile range (IQR) rule flags values far outside the middle 50%.
A flagged value is not automatically wrong; investigate it. The synthetic data
has no negative prices, but it may have expensive outliers.

In [6]:
q1, q3 = df["sale_price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["sale_price"] < lower) | (df["sale_price"] > upper)]
print("IQR bounds:", round(lower, 0), round(upper, 0))
print("outlier rows:", len(outliers))
print("negative prices:", int((df["sale_price"] <= 0).sum()))
assert (df["sale_price"] > 0).all()

IQR bounds: 154652.0 477448.0
outlier rows: 4
negative prices: 0


## 5. Write observations, not just plots

An EDA note should state what you saw and what it implies for modeling. Use this
template:

1. **Target:** sale price is right-skewed with a long upper tail.
2. **Strong signal:** area has the strongest positive linear correlation.
3. **Group effect:** neighbourhood medians differ, so include it as a feature.
4. **Risk:** expensive outliers may dominate squared-error metrics; report MAE too.
5. **Next check:** verify no target-derived columns exist before modeling.

A plot without an observation is decoration. Write the sentence.

**Next:** F07 - Supervised and unsupervised learning.